## 准备工作：导入所需库

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

## 2 序列模型

### 2.1 理论计算题

给定一个字符序列 "ababc"，假设采用一阶马尔可夫模型（即 $p(x_t | x_{t-1})$），使用拉普拉斯平滑（加1平滑）估计以下条件概率：

1. $p('a' | 'b')$
2. $p('c' | 'b')$

（词汇表为 `{'a','b','c'}`，计算时考虑所有可能转移，包括未出现的情况。）

In [3]:
print("=" * 50)
print("2.1 理论计算题：一阶马尔可夫模型 + 拉普拉斯平滑")
print("=" * 50)

seq = "ababc"
vocab = ['a', 'b', 'c']
V = len(vocab)  # 词汇表大小 = 3

print(f"序列: '{seq}'")
print(f"词汇表: {vocab}, V = {V}")
print()

# 统计转移计数
# 序列中的转移对: a->b, b->a, a->b, b->c
transitions = {}
for i in range(len(seq) - 1):
    pair = (seq[i], seq[i+1])
    transitions[pair] = transitions.get(pair, 0) + 1

print("转移对统计:")
for pair, count in sorted(transitions.items()):
    print(f"  {pair[0]} -> {pair[1]}: {count} 次")

# 统计每个字符的总出现次数（作为当前字符，即转移起点）
char_count = {}
for ch in vocab:
    char_count[ch] = sum(1 for i in range(len(seq) - 1) if seq[i] == ch)

print(f"\n各字符作为转移起点的次数:")
for ch in vocab:
    print(f"  '{ch}': {char_count[ch]} 次")

print("\n" + "-" * 40)
print("拉普拉斯平滑公式:")
print("  p(w_j | w_i) = (count(w_i -> w_j) + 1) / (count(w_i) + V)")
print("-" * 40)

# 1. p('a' | 'b')
count_b_to_a = transitions.get(('b', 'a'), 0)
count_b = char_count['b']
p_a_given_b = (count_b_to_a + 1) / (count_b + V)

print(f"\n1. p('a' | 'b'):")
print(f"   count(b -> a) = {count_b_to_a}")
print(f"   count(b) = {count_b}  (b 在 'ababc' 中作为起点出现 {count_b} 次)")
print(f"   p('a' | 'b') = ({count_b_to_a} + 1) / ({count_b} + {V})")
print(f"                = {count_b_to_a + 1} / {count_b + V}")
print(f"                = {p_a_given_b:.4f}")

# 2. p('c' | 'b')
count_b_to_c = transitions.get(('b', 'c'), 0)
p_c_given_b = (count_b_to_c + 1) / (count_b + V)

print(f"\n2. p('c' | 'b'):")
print(f"   count(b -> c) = {count_b_to_c}")
print(f"   count(b) = {count_b}")
print(f"   p('c' | 'b') = ({count_b_to_c} + 1) / ({count_b} + {V})")
print(f"                = {count_b_to_c + 1} / {count_b + V}")
print(f"                = {p_c_given_b:.4f}")

# 验证：从 b 出发的所有转移概率之和应为 1
p_b_given_b = (transitions.get(('b', 'b'), 0) + 1) / (count_b + V)
print(f"\n验证 (b->a, b->b, b->c 概率之和):")
print(f"  p('a'|'b') + p('b'|'b') + p('c'|'b') = {p_a_given_b:.4f} + {p_b_given_b:.4f} + {p_c_given_b:.4f} = {p_a_given_b + p_b_given_b + p_c_given_b:.4f} (应为 1.0)")

# 完整转移概率矩阵
print(f"\n完整转移概率矩阵（拉普拉斯平滑后）:")
print(f"        {'a':>8} {'b':>8} {'c':>8}")
for wi in vocab:
    probs = []
    for wj in vocab:
        cnt = transitions.get((wi, wj), 0)
        p = (cnt + 1) / (char_count[wi] + V)
        probs.append(f"{p:.4f}")
    print(f"  '{wi}'  " + "  ".join(probs))

2.1 理论计算题：一阶马尔可夫模型 + 拉普拉斯平滑
序列: 'ababc'
词汇表: ['a', 'b', 'c'], V = 3

转移对统计:
  a -> b: 2 次
  b -> a: 1 次
  b -> c: 1 次

各字符作为转移起点的次数:
  'a': 2 次
  'b': 2 次
  'c': 0 次

----------------------------------------
拉普拉斯平滑公式:
  p(w_j | w_i) = (count(w_i -> w_j) + 1) / (count(w_i) + V)
----------------------------------------

1. p('a' | 'b'):
   count(b -> a) = 1
   count(b) = 2  (b 在 'ababc' 中作为起点出现 2 次)
   p('a' | 'b') = (1 + 1) / (2 + 3)
                = 2 / 5
                = 0.4000

2. p('c' | 'b'):
   count(b -> c) = 1
   count(b) = 2
   p('c' | 'b') = (1 + 1) / (2 + 3)
                = 2 / 5
                = 0.4000

验证 (b->a, b->b, b->c 概率之和):
  p('a'|'b') + p('b'|'b') + p('c'|'b') = 0.4000 + 0.2000 + 0.4000 = 1.0000 (应为 1.0)

完整转移概率矩阵（拉普拉斯平滑后）:
               a        b        c
  'a'  0.2000  0.6000  0.2000
  'b'  0.4000  0.2000  0.4000
  'c'  0.3333  0.3333  0.3333


### 2.2 编程题

编写一个函数 `preprocess_text(text, n)`，完成以下步骤：
1. 将文本转换为小写，去除标点符号（保留字母和空格）。
2. 按空格分词。
3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）。
4. 用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签（用于自回归语言模型）。

返回词汇表字典和（特征列表, 标签列表）。

In [4]:
print("\n" + "=" * 50)
print("2.2 编程题：文本预处理函数")
print("=" * 50)

def preprocess_text(text, n):
    """
    文本预处理函数：构建词汇表并生成滑动窗口特征/标签。
    
    参数:
        text: 输入文本字符串
        n: 滑动窗口大小（上下文词数）
    
    返回:
        vocab: dict, {词: 整数ID}，按频率排序
        (features, labels): 特征列表和标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    tokens = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    # 按频率降序排列，频率相同则按字母序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成长度为 n 的特征序列和下一个词标签
    features = []
    labels = []
    for i in range(len(tokens) - n):
        window = tokens[i:i + n]  # n 个上下文词
        next_word = tokens[i + n]  # 下一个词（预测目标）
        features.append(window)
        labels.append(next_word)
    
    return vocab, (features, labels)


# ─── 测试 ───────────────────────────────────────────────
print("测试 preprocess_text 函数:")
print("-" * 40)

# 测试1: 示例输入
test_text = "The time machine"
n = 2
vocab, (features, labels) = preprocess_text(test_text, n)

print(f"\n1) 输入文本: \"{test_text}\", n={n}")
print(f"   词汇表: {vocab}")
print(f"   特征列表: {features}")
print(f"   标签列表: {labels}")

# 测试2: 更复杂的文本
test_text2 = "The quick brown fox jumps over the lazy dog. The dog was not amused!"
n2 = 3
vocab2, (features2, labels2) = preprocess_text(test_text2, n2)

print(f"\n2) 输入文本: \"{test_text2}\", n={n2}")
print(f"   词汇表大小: {len(vocab2)}")
print(f"   词汇表: {vocab2}")
print(f"   特征数量: {len(features2)}")
for i in range(min(5, len(features2))):
    print(f"     特征[{i}]: {features2[i]} -> 标签[{i}]: {labels2[i]}")

# 测试3: 包含标点符号和大写的文本
test_text3 = "Hello, World! Hello, Python! Python is great. Hello world."
n3 = 2
vocab3, (features3, labels3) = preprocess_text(test_text3, n3)

print(f"\n3) 输入文本: \"{test_text3}\", n={n3}")
clean_text = re.sub(r'[^a-z\s]', '', test_text3.lower())
print(f"   处理后: {clean_text}")
print(f"   词汇表（按频率排序）: {vocab3}")
print(f"   特征-标签对:")
for i, (feat, lab) in enumerate(zip(features3, labels3)):
    print(f"     {feat} -> '{lab}'")

print("\n说明：词汇表按词频降序排列（高频词获得较小ID），便于后续模型使用。")


2.2 编程题：文本预处理函数
测试 preprocess_text 函数:
----------------------------------------

1) 输入文本: "The time machine", n=2
   词汇表: {'machine': 0, 'the': 1, 'time': 2}
   特征列表: [['the', 'time']]
   标签列表: ['machine']

2) 输入文本: "The quick brown fox jumps over the lazy dog. The dog was not amused!", n=3
   词汇表大小: 11
   词汇表: {'the': 0, 'dog': 1, 'amused': 2, 'brown': 3, 'fox': 4, 'jumps': 5, 'lazy': 6, 'not': 7, 'over': 8, 'quick': 9, 'was': 10}
   特征数量: 11
     特征[0]: ['the', 'quick', 'brown'] -> 标签[0]: fox
     特征[1]: ['quick', 'brown', 'fox'] -> 标签[1]: jumps
     特征[2]: ['brown', 'fox', 'jumps'] -> 标签[2]: over
     特征[3]: ['fox', 'jumps', 'over'] -> 标签[3]: the
     特征[4]: ['jumps', 'over', 'the'] -> 标签[4]: lazy

3) 输入文本: "Hello, World! Hello, Python! Python is great. Hello world.", n=2
   处理后: hello world hello python python is great hello world
   词汇表（按频率排序）: {'hello': 0, 'python': 1, 'world': 2, 'great': 3, 'is': 4}
   特征-标签对:
     ['hello', 'world'] -> 'hello'
     ['world', 'hello'] -> 'pyth

## 3 循环神经网络

### 3.1 理论计算题

考虑一个线性RNN（无偏置），定义为 $h_t = W_{hh} h_{t-1} + W_{hx} x_t$，输出 $o_t = W_{oh} h_t$。假设损失函数为平方损失 $L = \frac{1}{2} \sum_{t=1}^{T} (o_t - y_t)^2$。推导损失对权重 $W_{hh}$ 的梯度表达式（通过时间反向传播，展开到所有时间步），并说明梯度消失或爆炸的条件。

In [5]:
print("\n" + "=" * 50)
print("3.1 理论计算题：BPTT 梯度推导")
print("=" * 50)

print("""
线性RNN定义:
  h_t = W_hh * h_{t-1} + W_hx * x_t    (无偏置)
  o_t = W_oh * h_t
  L   = (1/2) * sum_{t=1}^T (o_t - y_t)^2

============================================================
1. 通过时间反向传播 (BPTT) 推导 dL/dW_hh
============================================================

步骤1: 损失对输出 o_t 的梯度
  dL/do_t = o_t - y_t

步骤2: 输出对隐藏状态 h_t 的梯度
  do_t/dh_t = W_oh^T
  dL/dh_t = dL/do_t * do_t/dh_t = W_oh^T * (o_t - y_t)

  但 h_t 也影响后续所有时刻的损失（通过 h_{t+1}, h_{t+2}, ...）：
  dL/dh_t = W_oh^T * (o_t - y_t) + dL/dh_{t+1} * dh_{t+1}/dh_t

  其中 dh_{t+1}/dh_t = W_hh^T

  递归展开 (从 t=T 向 t=1 反向):
  dL/dh_T = W_oh^T * (o_T - y_T)
  dL/dh_{T-1} = W_oh^T * (o_{T-1} - y_{T-1}) + W_hh^T * dL/dh_T
  dL/dh_{T-2} = W_oh^T * (o_{T-2} - y_{T-2}) + W_hh^T * dL/dh_{T-1}
  ...

  一般形式:
  dL/dh_t = W_oh^T * (o_t - y_t)
          + sum_{k=t+1}^T (W_hh^T)^{k-t} * W_oh^T * (o_k - y_k)

步骤3: h_t 对 W_hh 的梯度
  h_t = W_hh * h_{t-1} + W_hx * x_t
  dh_t/dW_hh = h_{t-1}^T    (外积，形状同 W_hh)

步骤4: 链式法则汇总
  dL/dW_hh = sum_{t=1}^T dL/dh_t (外积) h_{t-1}

  其中 dL/dh_t 包含了从时刻 t 到 T 的所有反向传播的梯度。

============================================================
2. 梯度消失或爆炸的条件
============================================================

梯度中反复出现 W_hh^T 的幂次 (W_hh^T)^{k-t}。

对 W_hh 进行特征分解：W_hh = U * diag(lambda_i) * U^{-1}
则 (W_hh^T)^{k-t} = (U^T)^{-1} * diag(lambda_i^{k-t}) * U^T

  * 若 |lambda_max| < 1:
    -> 当 k-t 很大时，lambda_i^{k-t} -> 0
    -> 远期梯度趋近于 0 -> 梯度消失 (Vanishing Gradient)
    -> 模型无法学习长期依赖

  * 若 |lambda_max| > 1:
    -> 当 k-t 很大时，lambda_i^{k-t} -> 无穷大
    -> 远期梯度指数级增长 -> 梯度爆炸 (Exploding Gradient)
    -> 训练不稳定，参数更新剧烈

  * 若 |lambda_max| 约等于 1:
    -> 梯度可以较稳定地传播
    -> 有利于学习长期依赖

这就是原始RNN难以处理长序列的根本原因，
也是LSTM和GRU等门控机制被提出的动机。
""")


3.1 理论计算题：BPTT 梯度推导

线性RNN定义:
  h_t = W_hh * h_{t-1} + W_hx * x_t    (无偏置)
  o_t = W_oh * h_t
  L   = (1/2) * sum_{t=1}^T (o_t - y_t)^2

1. 通过时间反向传播 (BPTT) 推导 dL/dW_hh

步骤1: 损失对输出 o_t 的梯度
  dL/do_t = o_t - y_t

步骤2: 输出对隐藏状态 h_t 的梯度
  do_t/dh_t = W_oh^T
  dL/dh_t = dL/do_t * do_t/dh_t = W_oh^T * (o_t - y_t)

  但 h_t 也影响后续所有时刻的损失（通过 h_{t+1}, h_{t+2}, ...）：
  dL/dh_t = W_oh^T * (o_t - y_t) + dL/dh_{t+1} * dh_{t+1}/dh_t

  其中 dh_{t+1}/dh_t = W_hh^T

  递归展开 (从 t=T 向 t=1 反向):
  dL/dh_T = W_oh^T * (o_T - y_T)
  dL/dh_{T-1} = W_oh^T * (o_{T-1} - y_{T-1}) + W_hh^T * dL/dh_T
  dL/dh_{T-2} = W_oh^T * (o_{T-2} - y_{T-2}) + W_hh^T * dL/dh_{T-1}
  ...

  一般形式:
  dL/dh_t = W_oh^T * (o_t - y_t)
          + sum_{k=t+1}^T (W_hh^T)^{k-t} * W_oh^T * (o_k - y_k)

步骤3: h_t 对 W_hh 的梯度
  h_t = W_hh * h_{t-1} + W_hx * x_t
  dh_t/dW_hh = h_{t-1}^T    (外积，形状同 W_hh)

步骤4: 链式法则汇总
  dL/dW_hh = sum_{t=1}^T dL/dh_t (外积) h_{t-1}

  其中 dL/dh_t 包含了从时刻 t 到 T 的所有反向传播的梯度。

2. 梯度消失或爆炸的条件

梯度中反复出现 W_hh^T 的幂次 (W_hh^T)^{k-t}。

### 3.2 编程题

实现一个简单的 RNN 单元的前向传播和单步反向传播（仅计算梯度，不更新）。给定输入 $x_t$（形状 `(batch_size, input_size)`）、上一隐藏状态 $h_{prev}$（形状 `(batch_size, hidden_size)`），以及权重 $W_{hx}, W_{hh}, b_h$，计算当前隐藏状态 $h_t$。同时实现反向传播，已知上游梯度 `dh_next`（即损失对 $h_t$ 的梯度），计算 $dx_t, dh_{prev}, dW_{hx}, dW_{hh}, db_h$（使用 tanh 激活函数）。

In [6]:
print("\n" + "=" * 50)
print("3.2 编程题：RNN 单元前向与反向传播")
print("=" * 50)

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播：h_t = tanh(x_t @ W_hx + h_prev @ W_hh + b_h)
    
    参数:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 反向传播所需的中间值
    """
    # 线性变换
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h  # (batch_size, hidden_size)
    # tanh 激活
    h_t = np.tanh(a_t)
    # 缓存反向传播所需的值
    cache = (x_t, h_prev, a_t, h_t, W_hx, W_hh)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN 单元单步反向传播。
    
    参数:
        dh_next: 损失对 h_t 的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播时缓存的中间值
    
    返回:
        dx_t: 损失对 x_t 的梯度
        dh_prev: 损失对 h_{t-1} 的梯度
        dW_hx: 损失对 W_hx 的梯度
        dW_hh: 损失对 W_hh 的梯度
        db_h: 损失对 b_h 的梯度
    """
    x_t, h_prev, a_t, h_t, W_hx, W_hh = cache
    
    # tanh 的导数: d(tanh(z))/dz = 1 - tanh^2(z)
    da_t = dh_next * (1 - h_t ** 2)  # (batch_size, hidden_size)
    
    # 计算各参数的梯度
    dx_t = da_t @ W_hx.T           # (batch_size, input_size)
    dh_prev = da_t @ W_hh.T        # (batch_size, hidden_size)
    dW_hx = x_t.T @ da_t           # (input_size, hidden_size)
    dW_hh = h_prev.T @ da_t        # (hidden_size, hidden_size)
    db_h = da_t.sum(axis=0)        # (hidden_size,)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ─── 测试 ───────────────────────────────────────────────
print("\n测试 RNN 单元前向与反向传播:")
print("-" * 40)

np.random.seed(42)
batch_size, input_size, hidden_size = 3, 4, 5

# 随机初始化
x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)

# 前向传播
h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)

print(f"输入 x_t 形状: {x_t.shape}")
print(f"上一隐藏状态 h_prev 形状: {h_prev.shape}")
print(f"权重 W_hx 形状: {W_hx.shape}")
print(f"权重 W_hh 形状: {W_hh.shape}")
print(f"输出 h_t 形状: {h_t.shape}")
print(f"h_t 值范围: [{h_t.min():.4f}, {h_t.max():.4f}]")

# 反向传播
dh_next = np.random.randn(batch_size, hidden_size) * 0.1
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print(f"\n反向传播梯度形状检查:")
print(f"  dx_t 形状:     {dx_t.shape}  (期望: {x_t.shape}) ✓")
print(f"  dh_prev 形状:  {dh_prev.shape}  (期望: {h_prev.shape}) ✓")
print(f"  dW_hx 形状:    {dW_hx.shape}  (期望: {W_hx.shape}) ✓")
print(f"  dW_hh 形状:    {dW_hh.shape}  (期望: {W_hh.shape}) ✓")
print(f"  db_h 形状:     {db_h.shape}  (期望: {b_h.shape}) ✓")

# 数值梯度检查
print(f"\n数值梯度检查:")
epsilon = 1e-5
idx = (1, 2)

# 检查 W_hx[i,j]
W_hx_plus = W_hx.copy()
W_hx_plus[idx] += epsilon
h_t_plus, _ = rnn_cell_forward(x_t, h_prev, W_hx_plus, W_hh, b_h)
loss_plus = np.sum(h_t_plus)

W_hx_minus = W_hx.copy()
W_hx_minus[idx] -= epsilon
h_t_minus, _ = rnn_cell_forward(x_t, h_prev, W_hx_minus, W_hh, b_h)
loss_minus = np.sum(h_t_minus)

num_grad_W_hx = (loss_plus - loss_minus) / (2 * epsilon)

h_t_full, cache_full = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
_, _, dW_hx_full, _, _ = rnn_cell_backward(np.ones_like(h_t_full), cache_full)

print(f"  W_hx[{idx}]: 数值梯度 = {num_grad_W_hx:.8f}, 解析梯度 = {dW_hx_full[idx]:.8f}")
print(f"  差异 = {abs(num_grad_W_hx - dW_hx_full[idx]):.2e}")

# 检查 b_h[k]
b_h_plus = b_h.copy()
b_h_plus[2] += epsilon
h_t_plus2, _ = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h_plus)
loss_plus2 = np.sum(h_t_plus2)

b_h_minus = b_h.copy()
b_h_minus[2] -= epsilon
h_t_minus2, _ = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h_minus)
loss_minus2 = np.sum(h_t_minus2)

num_grad_bh = (loss_plus2 - loss_minus2) / (2 * epsilon)
_, _, _, _, db_h_full = rnn_cell_backward(np.ones_like(h_t_full), cache_full)
print(f"  b_h[2]: 数值梯度 = {num_grad_bh:.8f}, 解析梯度 = {db_h_full[2]:.8f}")
print(f"  差异 = {abs(num_grad_bh - db_h_full[2]):.2e}")

assert abs(num_grad_W_hx - dW_hx_full[idx]) < 1e-5, "W_hx 梯度检查失败！"
assert abs(num_grad_bh - db_h_full[2]) < 1e-5, "b_h 梯度检查失败！"
print("\n数值梯度检查通过 ✓")


3.2 编程题：RNN 单元前向与反向传播

测试 RNN 单元前向与反向传播:
----------------------------------------
输入 x_t 形状: (3, 4)
上一隐藏状态 h_prev 形状: (3, 5)
权重 W_hx 形状: (4, 5)
权重 W_hh 形状: (5, 5)
输出 h_t 形状: (3, 5)
h_t 值范围: [-0.5559, 0.2803]

反向传播梯度形状检查:
  dx_t 形状:     (3, 4)  (期望: (3, 4)) ✓
  dh_prev 形状:  (3, 5)  (期望: (3, 5)) ✓
  dW_hx 形状:    (4, 5)  (期望: (4, 5)) ✓
  dW_hh 形状:    (5, 5)  (期望: (5, 5)) ✓
  db_h 形状:     (5,)  (期望: (5,)) ✓

数值梯度检查:
  W_hx[(1, 2)]: 数值梯度 = 0.22664484, 解析梯度 = 0.22664484
  差异 = 9.69e-12
  b_h[2]: 数值梯度 = 2.63185386, 解析梯度 = 2.63185386
  差异 = 8.30e-11

数值梯度检查通过 ✓


## 4 高级循环神经网络

### 4.1 理论计算题

假设一个深度双向RNN，有 $L$ 层，每层隐藏单元数为 $H$，输入维度为 $D$，输出维度为 $O$（仅考虑最后输出层）。计算该模型的参数总数（包括所有全连接层的权重和偏置），忽略嵌入层和输出层之前的投影，明确给出表达式。

In [7]:
print("\n" + "=" * 50)
print("4.1 理论计算题：深度双向RNN参数计算")
print("=" * 50)

print("""
模型结构：L 层深度双向 RNN

每层包含一个前向 RNN 和一个后向 RNN。
除第一层外，每层的输入是上一层前向+后向隐藏状态的拼接 (维度 2H)。

============================================================
参数分解
============================================================

第 1 层:
  前向 RNN: W_hx^(1,f): H*D, W_hh^(1,f): H*H, b_h^(1,f): H
  后向 RNN: W_hx^(1,b): H*D, W_hh^(1,b): H*H, b_h^(1,b): H
  小计 = 2 * (HD + H^2 + H) = 2HD + 2H^2 + 2H

第 l 层 (l = 2, 3, ..., L):
  输入维度 = 2H (上一层双向拼接)
  前向 RNN: W_hx^(l,f): H*2H, W_hh^(l,f): H*H, b_h^(l,f): H
  后向 RNN: W_hx^(l,b): H*2H, W_hh^(l,b): H*H, b_h^(l,b): H
  小计 = 2 * (2H^2 + H^2 + H) = 2 * (3H^2 + H) = 6H^2 + 2H

输出层:
  W_ho: O * 2H  (输入为最后一层的双向拼接 2H)
  b_o: O
  小计 = 2HO + O

============================================================
总参数量
============================================================

Total = (2HD + 2H^2 + 2H) + (L-1)(6H^2 + 2H) + (2HO + O)

展开并化简:
  = 2HD + 2H^2 + 2H + 6H^2*L - 6H^2 + 2H*L - 2H + 2HO + O
  = 2HD + 2HO + 2HL + 2H^2(1 - 3) + 6H^2*L + (2H - 2H) + O
  = 2H(D + O + L) + 2H^2(3L - 2) + O

============================================================
特例验证 (L=2, H=128, D=100, O=10)
============================================================
""")

L, H, D, O = 2, 128, 100, 10

layer1_params = 2 * (H*D + H*H + H)
layer2_params = 2 * (2*H*H + H*H + H)
output_params = O * 2*H + O
total = layer1_params + layer2_params + output_params

formula_total = 2*H*(D + O + L) + 2*H*H*(3*L - 2) + O

print(f"  第1层参数: {layer1_params:,}")
print(f"  第2层参数: {layer2_params:,}")
print(f"  输出层参数: {output_params:,}")
print(f"  总参数 = {layer1_params:,} + {layer2_params:,} + {output_params:,} = {total:,}")
print(f"  公式验证: 2H(D+O+L) + 2H^2(3L-2) + O = {formula_total:,}")
print(f"  两者一致: {total == formula_total} ✓")

print(f"""
============================================================
结论
============================================================

总参数量 = 2H(D + O + L) + 2H^2(3L - 2) + O

参数主要由以下部分决定:
  - 输入嵌入相关参数: 2HD (仅第一层)
  - 层间连接参数: 6H^2(L-1) (随层数线性增长)
  - 循环连接参数: 2H^2 (每层每个方向)
  - 输出层参数: 2HO + O

最主要的是层间连接参数 6H^2(L-1)，与 H^2 成正比且随 L 线性增长。
""")


4.1 理论计算题：深度双向RNN参数计算

模型结构：L 层深度双向 RNN

每层包含一个前向 RNN 和一个后向 RNN。
除第一层外，每层的输入是上一层前向+后向隐藏状态的拼接 (维度 2H)。

参数分解

第 1 层:
  前向 RNN: W_hx^(1,f): H*D, W_hh^(1,f): H*H, b_h^(1,f): H
  后向 RNN: W_hx^(1,b): H*D, W_hh^(1,b): H*H, b_h^(1,b): H
  小计 = 2 * (HD + H^2 + H) = 2HD + 2H^2 + 2H

第 l 层 (l = 2, 3, ..., L):
  输入维度 = 2H (上一层双向拼接)
  前向 RNN: W_hx^(l,f): H*2H, W_hh^(l,f): H*H, b_h^(l,f): H
  后向 RNN: W_hx^(l,b): H*2H, W_hh^(l,b): H*H, b_h^(l,b): H
  小计 = 2 * (2H^2 + H^2 + H) = 2 * (3H^2 + H) = 6H^2 + 2H

输出层:
  W_ho: O * 2H  (输入为最后一层的双向拼接 2H)
  b_o: O
  小计 = 2HO + O

总参数量

Total = (2HD + 2H^2 + 2H) + (L-1)(6H^2 + 2H) + (2HO + O)

展开并化简:
  = 2HD + 2H^2 + 2H + 6H^2*L - 6H^2 + 2H*L - 2H + 2HO + O
  = 2HD + 2HO + 2HL + 2H^2(1 - 3) + 6H^2*L + (2H - 2H) + O
  = 2H(D + O + L) + 2H^2(3L - 2) + O

特例验证 (L=2, H=128, D=100, O=10)

  第1层参数: 58,624
  第2层参数: 98,560
  输出层参数: 2,570
  总参数 = 58,624 + 98,560 + 2,570 = 159,754
  公式验证: 2H(D+O+L) + 2H^2(3L-2) + O = 159,754
  两者一致: True ✓

结论

总参数量 = 2H(D + O + L) + 2H^

### 4.2 编程题

实现一个双向RNN编码器，接收序列 $X$（形状 `(seq_len, batch, input_dim)`），使用 `torch.nn.RNN` 或手动实现。要求返回每个时间步的拼接后的前向和后向隐藏状态（形状 `(seq_len, batch, 2*hidden_dim)`），以及最终时间步的拼接隐藏状态（作为序列表示）。

In [8]:
print("\n" + "=" * 50)
print("4.2 编程题：双向RNN编码器")
print("=" * 50)

class BiRNNEncoder(nn.Module):
    """
    双向 RNN 编码器。
    
    参数:
        input_dim: 输入特征维度
        hidden_dim: 隐藏单元数（每个方向）
        num_layers: RNN 层数，默认 1
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用 PyTorch 内置双向 RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False  # 输入形状 (seq_len, batch, input_dim)
        )
    
    def forward(self, X):
        """
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，
                     形状 (seq_len, batch, 2 * hidden_dim)
            final_h: 最终时间步的拼接隐藏状态，
                     形状 (batch, 2 * hidden_dim * num_layers)
        """
        seq_len, batch_size, _ = X.shape
        
        # RNN 前向传播
        # outputs: (seq_len, batch, 2 * hidden_dim)
        # h_n: (2 * num_layers, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 将 h_n 重排为 (batch, 2 * num_layers * hidden_dim)
        h_n = h_n.permute(1, 0, 2)  # (batch, 2*num_layers, hidden_dim)
        final_h = h_n.reshape(batch_size, -1)  # (batch, 2*num_layers*hidden_dim)
        
        return outputs, final_h


# ─── 测试 ───────────────────────────────────────────────
print("\n测试双向RNN编码器:")
print("-" * 40)

seq_len, batch_size, input_dim = 5, 2, 8
hidden_dim = 16

torch.manual_seed(42)
X = torch.randn(seq_len, batch_size, input_dim)

encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=1)

print(f"输入 X 形状: {X.shape}  (seq_len={seq_len}, batch={batch_size}, input_dim={input_dim})")

with torch.no_grad():
    outputs, final_h = encoder(X)

print(f"\n每个时间步的输出 outputs 形状: {outputs.shape}")
expected_output = torch.Size([seq_len, batch_size, 2*hidden_dim])
print(f"  期望: {tuple(expected_output)}")
print(f"  匹配: {outputs.shape == expected_output} ✓")

print(f"\n最终隐藏状态 final_h 形状: {final_h.shape}")
expected_final = torch.Size([batch_size, 2*hidden_dim])
print(f"  期望: {tuple(expected_final)}")
print(f"  匹配: {final_h.shape == expected_final} ✓")

# 验证双向拼接
print(f"\n前向 hidden_dim={hidden_dim} 维:")
print(f"  {outputs[-1, 0, :hidden_dim].numpy()[:4]}...")
print(f"后向 hidden_dim={hidden_dim} 维:")
print(f"  {outputs[-1, 0, hidden_dim:].numpy()[:4]}...")

# 测试多层
print(f"\n--- 多层双向RNN测试 ---")
encoder2 = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=2)
with torch.no_grad():
    outputs2, final_h2 = encoder2(X)

print(f"num_layers=2:")
print(f"  outputs 形状: {outputs2.shape}  (期望: [{seq_len}, {batch_size}, {32}])")
print(f"  final_h 形状: {final_h2.shape}  (期望: [{batch_size}, {64}])")
print(f"  匹配: {final_h2.shape == torch.Size([batch_size, 4*hidden_dim])} ✓")

total_params = sum(p.numel() for p in encoder.parameters())
print(f"\n编码器参数量: {total_params:,}")

print("\n说明：双向RNN编码器通过同时从前向和后向读取序列，")
print("能够捕获每个位置的完整上下文信息，广泛应用于序列标注、机器翻译等任务。")


4.2 编程题：双向RNN编码器

测试双向RNN编码器:
----------------------------------------
输入 X 形状: torch.Size([5, 2, 8])  (seq_len=5, batch=2, input_dim=8)

每个时间步的输出 outputs 形状: torch.Size([5, 2, 32])
  期望: (5, 2, 32)
  匹配: True ✓

最终隐藏状态 final_h 形状: torch.Size([2, 32])
  期望: (2, 32)
  匹配: True ✓

前向 hidden_dim=16 维:
  [-0.49672583 -0.44459555 -0.44931906 -0.68555623]...
后向 hidden_dim=16 维:
  [ 0.33809778 -0.39861813 -0.55108356  0.24256484]...

--- 多层双向RNN测试 ---
num_layers=2:
  outputs 形状: torch.Size([5, 2, 32])  (期望: [5, 2, 32])
  final_h 形状: torch.Size([2, 64])  (期望: [2, 64])
  匹配: True ✓

编码器参数量: 832

说明：双向RNN编码器通过同时从前向和后向读取序列，
能够捕获每个位置的完整上下文信息，广泛应用于序列标注、机器翻译等任务。


## 5 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$，使用负采样（采样 $K$ 个负样本）。推导其损失函数（对数似然）的表达式，并说明如何从噪声分布中采样负样本。假设词向量为 $v_c, u_o$，负样本词向量为 $u_{n_k}$，写出完整的目标函数。

In [9]:
print("\n" + "=" * 50)
print("5.1 理论计算题：Skip-gram 负采样损失函数")
print("=" * 50)

print("""
============================================================
Skip-gram 模型与负采样
============================================================

给定:
  - 中心词 w_c，其词向量为 v_c (维度 d)
  - 上下文词 w_o (正样本)，其输出向量为 u_o (维度 d)
  - K 个负样本词 {w_n1, ..., w_nK}，输出向量为 {u_n1, ..., u_nK}

============================================================
1. 目标函数推导
============================================================

对每个 (w_c, w_o) 正样本对，我们希望:
  - 最大化正样本的概率: P(D=1 | w_c, w_o) = sigma(u_o^T * v_c)
  - 最小化负样本的概率: 等价于最大化 P(D=0 | w_c, w_nk)
    P(D=0 | w_c, w_nk) = 1 - sigma(u_nk^T * v_c) = sigma(-u_nk^T * v_c)

其中 sigma(x) = 1/(1+e^{-x}) 是 sigmoid 函数。

单个样本对的负采样损失函数:

  L = -[ log sigma(u_o^T * v_c)                    <-- 正样本对数似然
       + sum_{k=1}^K log sigma(-u_nk^T * v_c) ]    <-- K个负样本对数似然

   = -[ log sigma(u_o^T * v_c)
       + sum_{k=1}^K log(1 - sigma(u_nk^T * v_c)) ]

完整语料库上的目标函数:

  J = sum_{(w_c, w_o) in D} [ log sigma(u_o^T * v_c)
       + sum_{k=1}^K E_{w_n ~ P_n(w)} [log sigma(-u_n^T * v_c)] ]

  其中 D 是所有 (中心词, 上下文词) 正样本对集合。

============================================================
2. 噪声分布与负采样策略
============================================================

噪声分布 P_n(w) 通常采用 unigram 分布的 3/4 次幂:

   P_n(w) = freq(w)^{3/4} / sum_{w'} freq(w')^{3/4}

为什么用 3/4 次幂？
  - 原始分布中高频词被采样概率过高，低频词几乎不会被采样
  - 3/4 次幂起到了"平滑"作用，提高了低频词的采样概率
  - 例如: freq("the") 远大于 freq("zebra")
    * 原始: P("the") / P("zebra") 可能 > 1000
    * 3/4 次幂后: 差距缩小到约 100 倍
  - 这使得模型也能为低频词学习有意义的表示

实际采样步骤:
  1. 计算每个词 w 的 freq(w)^{3/4}
  2. 归一化得到概率分布
  3. 根据该分布独立采样 K 个词作为负样本
  4. 通常排除正样本上下文词（如果恰好采样到）

============================================================
3. 最终目标函数（完整形式）
============================================================

对一对中心词-上下文词 (w_c, w_o):

  log-likelihood = log sigma(u_o^T * v_c)
                 + sum_{k=1}^K log sigma(-u_nk^T * v_c)
  loss = -log-likelihood

训练时最小化该损失，同时更新:
  - 中心词向量 v_c (输入嵌入)
  - 上下文词向量 u_o (正样本输出嵌入)
  - 负样本词向量 u_nk (负样本输出嵌入)
""")


5.1 理论计算题：Skip-gram 负采样损失函数

Skip-gram 模型与负采样

给定:
  - 中心词 w_c，其词向量为 v_c (维度 d)
  - 上下文词 w_o (正样本)，其输出向量为 u_o (维度 d)
  - K 个负样本词 {w_n1, ..., w_nK}，输出向量为 {u_n1, ..., u_nK}

1. 目标函数推导

对每个 (w_c, w_o) 正样本对，我们希望:
  - 最大化正样本的概率: P(D=1 | w_c, w_o) = sigma(u_o^T * v_c)
  - 最小化负样本的概率: 等价于最大化 P(D=0 | w_c, w_nk)
    P(D=0 | w_c, w_nk) = 1 - sigma(u_nk^T * v_c) = sigma(-u_nk^T * v_c)

其中 sigma(x) = 1/(1+e^{-x}) 是 sigmoid 函数。

单个样本对的负采样损失函数:

  L = -[ log sigma(u_o^T * v_c)                    <-- 正样本对数似然
       + sum_{k=1}^K log sigma(-u_nk^T * v_c) ]    <-- K个负样本对数似然

   = -[ log sigma(u_o^T * v_c)
       + sum_{k=1}^K log(1 - sigma(u_nk^T * v_c)) ]

完整语料库上的目标函数:

  J = sum_{(w_c, w_o) in D} [ log sigma(u_o^T * v_c)
       + sum_{k=1}^K E_{w_n ~ P_n(w)} [log sigma(-u_n^T * v_c)] ]

  其中 D 是所有 (中心词, 上下文词) 正样本对集合。

2. 噪声分布与负采样策略

噪声分布 P_n(w) 通常采用 unigram 分布的 3/4 次幂:

   P_n(w) = freq(w)^{3/4} / sum_{w'} freq(w')^{3/4}

为什么用 3/4 次幂？
  - 原始分布中高频词被采样概率过高，低频词几乎不会被采样
  - 3/4 次幂起到了"平滑"作用，提高了低频词的采样概率
  -

### 5.2 编程题

实现 CBOW 模型的前向传播和损失计算（不使用负采样，仅用完整 softmax）。给定一批上下文词的索引列表（每个样本有 `context_size` 个上下文词），词汇表大小 $V$，嵌入维度 $d$。输入权重矩阵 $W$（形状 `(V, d)`）和输出权重矩阵 $W_{out}$（形状 `(d, V)`）。计算平均上下文向量作为隐藏层，然后计算输出概率分布，并计算交叉熵损失（目标为中心词索引）。返回损失值。

In [10]:
print("\n" + "=" * 50)
print("5.2 编程题：CBOW 模型前向传播与损失计算")
print("=" * 50)

def cbow_forward_and_loss(context_indices, center_indices, W, W_out):
    """
    CBOW 模型前向传播和交叉熵损失计算。
    
    CBOW 思想: 用上下文词的平均嵌入来预测中心词。
    
    参数:
        context_indices: list of list，每个内层列表包含 context_size 个上下文词索引
        center_indices: list，中心词索引列表
        W: 输入嵌入矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 标量交叉熵损失
        probs: softmax 概率分布，形状 (batch_size, V)
    """
    batch_size = len(context_indices)
    d = W.shape[1]
    V = W_out.shape[1]
    
    # 1. 查找上下文词的嵌入向量并取平均 -> 隐藏层
    # h 形状: (batch_size, d)
    h = np.zeros((batch_size, d))
    for i, ctx in enumerate(context_indices):
        # ctx 是 context_size 个索引的列表
        context_embeddings = W[ctx]  # (context_size, d)
        h[i] = np.mean(context_embeddings, axis=0)  # (d,)
    
    # 2. 计算输出 logits
    # logits = h @ W_out, 形状 (batch_size, V)
    logits = h @ W_out  # (batch_size, V)
    
    # 3. Softmax（数值稳定版本）
    logits_max = np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits - logits_max)
    probs = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
    
    # 4. 交叉熵损失
    # 对每个样本，取目标中心词对应的 -log 概率
    correct_probs = probs[np.arange(batch_size), center_indices]
    loss = -np.mean(np.log(correct_probs + 1e-12))  # 加小值防止 log(0)
    
    return loss, probs


# ─── 测试 ───────────────────────────────────────────────
print("\n测试 CBOW 模型:")
print("-" * 40)

np.random.seed(42)

# 模拟数据
V = 100      # 词汇表大小
d = 50       # 嵌入维度
batch_size = 4
context_size = 3

# 随机初始化嵌入矩阵
W = np.random.randn(V, d) * 0.1
W_out = np.random.randn(d, V) * 0.1

# 模拟上下文词索引和中心词索引
context_indices = [
    [2, 5, 8],
    [1, 3, 7],
    [4, 6, 9],
    [0, 2, 3],
]
center_indices = [10, 15, 20, 5]

print(f"词汇表大小 V = {V}")
print(f"嵌入维度 d = {d}")
print(f"批次大小 = {batch_size}, 上下文窗口 = {context_size}")
print(f"\n上下文词索引: {context_indices}")
print(f"中心词索引: {center_indices}")

# 计算损失
loss, probs = cbow_forward_and_loss(context_indices, center_indices, W, W_out)

print(f"\n输出概率分布形状: {probs.shape} (batch_size={batch_size}, V={V})")
print(f"交叉熵损失: {loss:.6f}")

# 验证：概率分布的和应为 1
prob_sums = probs.sum(axis=1)
print(f"每个样本概率和: {prob_sums} (应接近 1.0)")

# 验证：正确类别的概率
for i in range(batch_size):
    print(f"  样本 {i}: 中心词={center_indices[i]}, "
          f"预测概率={probs[i, center_indices[i]]:.6f}, "
          f"最大概率类别={np.argmax(probs[i])}")

# 均匀分布参考
print(f"\n均匀分布下的期望损失: -ln(1/V) = -ln(1/{V}) = {np.log(V):.4f}")
print(f"当前随机初始化的损失: {loss:.6f}")
print(f"(实际训练中，损失会随着训练逐渐降低)")

print(f"\n说明：CBOW 通过上下文词的平均嵌入预测中心词，")
print(f"训练后相似的词会获得相近的嵌入向量。")


5.2 编程题：CBOW 模型前向传播与损失计算

测试 CBOW 模型:
----------------------------------------
词汇表大小 V = 100
嵌入维度 d = 50
批次大小 = 4, 上下文窗口 = 3

上下文词索引: [[2, 5, 8], [1, 3, 7], [4, 6, 9], [0, 2, 3]]
中心词索引: [10, 15, 20, 5]

输出概率分布形状: (4, 100) (batch_size=4, V=100)
交叉熵损失: 4.577583
每个样本概率和: [1. 1. 1. 1.] (应接近 1.0)
  样本 0: 中心词=10, 预测概率=0.010451, 最大概率类别=73
  样本 1: 中心词=15, 预测概率=0.010014, 最大概率类别=24
  样本 2: 中心词=20, 预测概率=0.010760, 最大概率类别=8
  样本 3: 中心词=5, 预测概率=0.009916, 最大概率类别=83

均匀分布下的期望损失: -ln(1/V) = -ln(1/100) = 4.6052
当前随机初始化的损失: 4.577583
(实际训练中，损失会随着训练逐渐降低)

说明：CBOW 通过上下文词的平均嵌入预测中心词，
训练后相似的词会获得相近的嵌入向量。


## 6 注意力机制

### 6.1 理论计算题

给定查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$，键矩阵 $K \in \mathbb{R}^{3 \times 4}$，值矩阵 $V \in \mathbb{R}^{3 \times 5}$。计算缩放点积注意力（无掩码）的输出矩阵，要求写出中间步骤（先计算得分矩阵，再 softmax，再加权求和）。使用 $\text{score} = QK^T / \sqrt{d_k}$（$d_k = 4$）。

In [11]:
print("\n" + "=" * 50)
print("6.1 理论计算题：缩放点积注意力计算")
print("=" * 50)

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

# 随机生成 Q, K, V（使用具体数值方便演示）
Q = np.array([
    [0.5, -0.2, 0.1, 0.8],
    [-0.3, 0.6, -0.5, 0.2]
])  # (2, 4)

K = np.array([
    [0.3, 0.7, -0.1, 0.4],
    [-0.6, 0.2, 0.9, -0.3],
    [0.1, -0.5, 0.2, 0.6]
])  # (3, 4)

V = np.array([
    [1.0, 0.0, 0.5, -0.2, 0.3],
    [0.2, 0.8, -0.1, 0.5, -0.4],
    [-0.3, 0.3, 0.7, 0.1, 0.6]
])  # (3, 5)

print("输入矩阵:")
print(f"Q 形状 {Q.shape}:")
print(Q)
print(f"\nK 形状 {K.shape}:")
print(K)
print(f"\nV 形状 {V.shape}:")
print(V)

d_k = Q.shape[1]  # = 4

# 步骤1: 计算得分矩阵 scores = Q @ K^T / sqrt(d_k)
print(f"\n{'='*50}")
print(f"步骤1: 计算得分矩阵")
print(f"  scores = Q @ K^T / sqrt(d_k)  (d_k={d_k})")
scores = Q @ K.T / np.sqrt(d_k)
print(f"\n  Q @ K^T:")
print(Q @ K.T)
print(f"\n  scores = (Q @ K^T) / sqrt({d_k}) = (Q @ K^T) / {np.sqrt(d_k):.4f}:")
print(f"  scores 形状: {scores.shape}")
print(scores)

# 步骤2: Softmax（沿每行）
print(f"\n{'='*50}")
print(f"步骤2: Softmax 归一化（沿 key 维度，即每行）")
print(f"  weights = softmax(scores, dim=-1)")

def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

weights = softmax(scores, axis=-1)
print(f"\n  weights 形状: {weights.shape}")
print(weights)
print(f"\n  每行权重和: {weights.sum(axis=1)} (应全为1.0)")

# 步骤3: 加权求和 output = weights @ V
print(f"\n{'='*50}")
print(f"步骤3: 加权求和")
print(f"  output = weights @ V")
output = weights @ V
print(f"\n  output 形状: {output.shape} (应为 2x5)")
print(output)

print(f"\n{'='*50}")
print("计算结果汇总:")
print(f"  输入: Q(2x4), K(3x4), V(3x5)")
print(f"  得分矩阵 scores: (2x3)")
print(f"  注意力权重 weights: (2x3)")
print(f"  输出 output: (2x5)")
print(f"\n缩放因子 1/sqrt(d_k) = 1/{np.sqrt(d_k):.4f} = {1/np.sqrt(d_k):.4f}")
print(f"防止内积过大导致 softmax 梯度消失。")


6.1 理论计算题：缩放点积注意力计算
输入矩阵:
Q 形状 (2, 4):
[[ 0.5 -0.2  0.1  0.8]
 [-0.3  0.6 -0.5  0.2]]

K 形状 (3, 4):
[[ 0.3  0.7 -0.1  0.4]
 [-0.6  0.2  0.9 -0.3]
 [ 0.1 -0.5  0.2  0.6]]

V 形状 (3, 5):
[[ 1.   0.   0.5 -0.2  0.3]
 [ 0.2  0.8 -0.1  0.5 -0.4]
 [-0.3  0.3  0.7  0.1  0.6]]

步骤1: 计算得分矩阵
  scores = Q @ K^T / sqrt(d_k)  (d_k=4)

  Q @ K^T:
[[ 0.32 -0.49  0.65]
 [ 0.46 -0.21 -0.31]]

  scores = (Q @ K^T) / sqrt(4) = (Q @ K^T) / 2.0000:
  scores 形状: (2, 3)
[[ 0.16  -0.245  0.325]
 [ 0.23  -0.105 -0.155]]

步骤2: Softmax 归一化（沿 key 维度，即每行）
  weights = softmax(scores, dim=-1)

  weights 形状: (2, 3)
[[0.3513 0.2343 0.4143]
 [0.4174 0.2986 0.284 ]]

  每行权重和: [1. 1.] (应全为1.0)

步骤3: 加权求和
  output = weights @ V

  output 形状: (2, 5) (应为 2x5)
[[0.2739 0.3118 0.4423 0.0883 0.2603]
 [0.3919 0.3241 0.3777 0.0942 0.1762]]

计算结果汇总:
  输入: Q(2x4), K(3x4), V(3x5)
  得分矩阵 scores: (2x3)
  注意力权重 weights: (2x3)
  输出 output: (2x5)

缩放因子 1/sqrt(d_k) = 1/2.0000 = 0.5000
防止内积过大导致 softmax 梯度消失。


### 6.2 编程题

实现多头注意力（Multi-Head Attention）的前向传播，假设 `num_heads=2`，`d_model=4`。给定输入 $X$（形状 `(seq_len, batch, d_model)`），分别线性投影得到 $Q, K, V$（每个头的维度 $d_k = d_v = d_{model} / num\_heads$）。对每个头计算缩放点积注意力，然后将所有头的输出拼接并经过最终线性层。返回输出（形状与输入相同）。

In [12]:
print("\n" + "=" * 50)
print("6.2 编程题：多头注意力（Multi-Head Attention）")
print("=" * 50)

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制。
    
    参数:
        d_model: 模型维度，默认 4
        num_heads: 注意力头数，默认 2
    """
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        """
        参数:
            X: 输入，形状 (seq_len, batch, d_model)
        
        返回:
            output: 多头注意力输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 拆分为多头
        # 将 d_model 维度拆为 (num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k)
        
        # 3. 转置为 (batch, num_heads, seq_len, d_k) 便于批量注意力计算
        Q = Q.permute(1, 2, 0, 3)  # (batch, num_heads, seq_len, d_k)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 4. 缩放点积注意力
        scale = self.d_k ** 0.5
        scores = Q @ K.transpose(-2, -1) / scale  # (batch, num_heads, seq_len, seq_len)
        attn_weights = F.softmax(scores, dim=-1)   # (batch, num_heads, seq_len, seq_len)
        attn_output = attn_weights @ V              # (batch, num_heads, seq_len, d_k)
        
        # 5. 拼接多头输出
        # (batch, num_heads, seq_len, d_k) -> (seq_len, batch, num_heads, d_k)
        attn_output = attn_output.permute(2, 0, 1, 3)
        # 合并最后两维: (seq_len, batch, d_model)
        attn_output = attn_output.reshape(seq_len, batch_size, self.d_model)
        
        # 6. 最终线性投影
        output = self.W_o(attn_output)
        
        return output


# ─── 测试 ───────────────────────────────────────────────
print("\n测试多头注意力 (num_heads=2, d_model=4):")
print("-" * 40)

torch.manual_seed(42)

seq_len, batch_size, d_model = 3, 2, 4
num_heads = 2

# 创建随机输入
X = torch.randn(seq_len, batch_size, d_model)
print(f"输入 X 形状: {X.shape}  (seq_len={seq_len}, batch={batch_size}, d_model={d_model})")

# 创建多头注意力模块
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
print(f"多头注意力的配置: num_heads={num_heads}, d_model={d_model}, d_k={d_model // num_heads}")

with torch.no_grad():
    output = mha(X)

print(f"\n输出形状: {output.shape}")
print(f"  期望: ({seq_len}, {batch_size}, {d_model})")
print(f"  匹配: {output.shape == X.shape} ✓")

# 深入检查中间步骤
print(f"\n--- 中间步骤详情 ---")
print(f"每个头的维度 d_k = d_model / num_heads = {d_model} / {num_heads} = {d_model // num_heads}")
print(f"Q, K, V 投影后形状: ({seq_len}, {batch_size}, {d_model})")
print(f"拆分多头后形状: ({seq_len}, {batch_size}, {num_heads}, {d_model // num_heads})")
print(f"注意力权重形状: ({batch_size}, {num_heads}, {seq_len}, {seq_len})")

# 计算参数量
total_params = sum(p.numel() for p in mha.parameters())
print(f"\n多头注意力总参数量: {total_params}")
print(f"  W_q: {d_model}x{d_model} = {d_model * d_model}")
print(f"  W_k: {d_model}x{d_model} = {d_model * d_model}")
print(f"  W_v: {d_model}x{d_model} = {d_model * d_model}")
print(f"  W_o: {d_model}x{d_model} = {d_model * d_model}")
print(f"  总计: 4 x {d_model * d_model} = {4 * d_model * d_model}")

# 与 PyTorch 内置实现对比
mha_builtin = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, batch_first=False)
print(f"\n与 PyTorch 内置 nn.MultiheadAttention 对比:")
print(f"  内置实现参数量: {sum(p.numel() for p in mha_builtin.parameters())}")
print(f"  自定义实现参数量: {total_params}")
print(f"  (内置实现包含 bias，所以参数量略有不同)")

print(f"\n说明：多头注意力让模型同时关注不同表示子空间的信息，")
print(f"是 Transformer 架构的核心组件。")


6.2 编程题：多头注意力（Multi-Head Attention）

测试多头注意力 (num_heads=2, d_model=4):
----------------------------------------
输入 X 形状: torch.Size([3, 2, 4])  (seq_len=3, batch=2, d_model=4)
多头注意力的配置: num_heads=2, d_model=4, d_k=2

输出形状: torch.Size([3, 2, 4])
  期望: (3, 2, 4)
  匹配: True ✓

--- 中间步骤详情 ---
每个头的维度 d_k = d_model / num_heads = 4 / 2 = 2
Q, K, V 投影后形状: (3, 2, 4)
拆分多头后形状: (3, 2, 2, 2)
注意力权重形状: (2, 2, 3, 3)

多头注意力总参数量: 64
  W_q: 4x4 = 16
  W_k: 4x4 = 16
  W_v: 4x4 = 16
  W_o: 4x4 = 16
  总计: 4 x 16 = 64

与 PyTorch 内置 nn.MultiheadAttention 对比:
  内置实现参数量: 80
  自定义实现参数量: 64
  (内置实现包含 bias，所以参数量略有不同)

说明：多头注意力让模型同时关注不同表示子空间的信息，
是 Transformer 架构的核心组件。
